In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/mSHIFT_SHeS/code_ocean/

/content/drive/MyDrive/mSHIFT_SHeS/code_ocean


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys

In [ ]:
# Add the parent directory of the notebook to sys.path, enables module imports from analysis_code
sys.path.append(str(Path().resolve() / 'code/notebooks/notebook_code'))

This notebook takes the output from the 2022 Nutrient Databank (NDB) to foodDB mapping and their associated impacts and performs some additional preprocessing steps. The output is a dataset of ~3000 food items and the associated impacts and cost of each item per 100g from the foodDB data. Detail of the methodology to develop the mapping will be provided in an upcoming paper.

The primary data used to in htis notebook are not publically available but the code is provided for the purposes of transparaency. The resultant dataset of the impacts and costs of items in the NDB are publically available at https://doi.org/10.7488/ds/8131

Import necessary libraries

In [ ]:
import numpy as np
import pandas as pd
import time
import os
import sys
from pathlib import Path


Load funcitons to process the price data

In [ ]:
from data_processing import price_error, price_variable

In [ ]:
!ls

capsule_extraction.ipynb   MET_distribution_moderate_PA.png
code			   MET_distribution_sedentary_nonsleep.png
data			   mvpa_recalibration.png
df_weight_sense_check.csv  PAL_distribution_adusted_SHeS.png
environment		   REPRODUCING.md
metadata		   results


In [ ]:
data_path = Path("data")

Load the updated pre-processed env impact data per 100g

In [ ]:
df_impacts_raw = pd.read_csv(data_path / "foodDB_data_processing/Env_Data_Linkage_2025-02-19.csv")

In [ ]:
df_impacts_raw.set_index('ndb_cat', inplace=True)

Load price data

In [ ]:
# Average price over all years --> Not used other than for maintaining helpful data struture for deriving the NDB price data
df_price = pd.read_csv(data_path / "foodDB_data_processing/Updated_Price_Data_With_Exclusions_2025-04-11.csv")

# Year specific price estimates for each item, excluding all items in the mapping that have a price >500p/100g
df_price_year = pd.read_csv(data_path / "foodDB_data_processing/Updated_Price_Data_With_Exclusions_Year_By_Year2025-04-11.csv")

In [ ]:
df_price_year.head()

,ndb_cat,year,num_unique_products,tot_num_observations,sd_size,sd_price,sd_median_size,sd_median_price,mean_price,mean_size,median_price,median_size
0,"50/50 bread (e.g. Kingsmill, Warburtons)",2019,6,6,0.000000,0.000000,210.752620,10.767114,21.561111,591.666667,21.561111,591.666667
1,"50/50 bread (e.g. Kingsmill, Warburtons)",2021,8,8,0.000000,0.000000,175.127505,10.727291,19.531250,681.250000,19.531250,681.250000
2,"50/50 bread (e.g. Kingsmill, Warburtons)",2022,7,7,0.000000,0.000000,207.880460,12.888949,23.000000,621.428571,23.000000,621.428571
3,7 Up free / light,2019,3,4,0.000000,0.000000,11.547005,1.925985,8.200337,1993.333333,8.200337,1993.333333
4,7 Up free / light,2021,20,27,95.249891,8.876734,2084.948109,49.822146,35.866856,2166.250000,31.503977,2155.750000


Extract datasets by the year

In [ ]:
df_price_2022 = df_price_year[df_price_year['year']==2022].copy()
df_price_2021 = df_price_year[df_price_year['year']==2021].copy()
df_price_2019 = df_price_year[df_price_year['year']==2019].copy()

Merge the price data from each year into single dataframe and combine with impact data

In [ ]:
# Reset index for dataframes that need it
df_price.set_index('ndb_cat', inplace=True)
df_price_2022 = df_price_2022.set_index('ndb_cat')
df_price_2021 = df_price_2021.set_index('ndb_cat')
df_price_2019 = df_price_2019.set_index('ndb_cat')

# Merge the dataframes
df_merged = pd.merge(df_impacts_raw, df_price, left_index=True, right_index=True, how='outer')
df_merged = pd.merge(df_merged, df_price_2022, left_index=True, right_index=True, how='outer', suffixes=('', '_2022'))
df_merged = pd.merge(df_merged, df_price_2021, left_index=True, right_index=True, how='outer', suffixes=('', '_2021'))
df_merged = pd.merge(df_merged, df_price_2019, left_index=True, right_index=True, how='outer', suffixes=('', '_2019'))

In [ ]:
df_impacts = df_merged.copy()

In [ ]:
df_impacts['mean_price_sim'] = df_impacts.apply(lambda row: price_variable(row, metric='mean'), axis=1)
df_impacts['median_price_sim'] = df_impacts.apply(lambda row: price_variable(row, metric='median'), axis=1)
df_impacts['sd_price_sim'] = df_impacts.apply(lambda row: price_error(row), axis=1)

In [ ]:
env_columns = np.loadtxt(data_path / "indicator_lists/env_columns.txt", dtype = str).tolist()
mean_env_columns = np.loadtxt(data_path / "indicator_lists/mean_env_columns.txt", dtype = str).tolist()
error_columns = np.loadtxt(data_path / "indicator_lists/error_columns.txt", dtype = str).tolist()
price_columns = np.loadtxt(data_path / "indicator_lists/price_columns.txt", dtype = str).tolist()

Some items with the same food code do not have a corresponding descirition in SHeS. Take average over items that we do have which have the same food code

In [ ]:
df_impacts.loc['Savoury pastry (e.g. cheese pastry)', mean_env_columns] = df_impacts.loc[['Cheese and onion pasty/roll (includes potato)','Cheese pastry (e.g. straw/twist)'], mean_env_columns].mean(axis=0)
df_impacts.loc['Savoury pastry (e.g. cheese pastry)', error_columns] = ((df_impacts.loc[['Cheese and onion pasty/roll (includes potato)','Cheese pastry (e.g. straw/twist)'], error_columns]**2).sum(axis=0))**0.5/2

In [ ]:
df_impacts.loc['Oat milk', mean_env_columns] = df_impacts.loc[['Oat milk/drink, flavoured (e.g. Alpro chocolate)','Oat milk/drink (e.g. Alpro, Oatly)'], mean_env_columns].mean(axis=0)
df_impacts.loc['Oat milk', error_columns] = ((df_impacts.loc[['Oat milk/drink, flavoured (e.g. Alpro chocolate)','Oat milk/drink (e.g. Alpro, Oatly)'], error_columns]**2).sum(axis=0))**0.5/2

Manually include the water use associated with tap water

In [ ]:
df_impacts.loc['Water (from tap, including hot water, filtered water)', :] = 0

In [ ]:
# Units of water use in L, 0.1L/100g of water.

df_impacts.loc['Water (from tap, including hot water, filtered water)', 'mean_WaterUse'] = 0.1
df_impacts.loc['Water (from tap, including hot water, filtered water)', 'median_WaterUse'] = 0.1

Price data for bananas errouneosly set by pack size -> divide by 5 for better approximation of price per banana

In [ ]:
df_impacts.loc['Banana', price_columns] /= 5

Save the dataset as a .parquet

In [ ]:
df_impacts.to_parquet(data_path / "df_impacts_per_100g_1902.parquet")

In [ ]:
df_impacts.to_excel(data_path / "df_impacts_per_100g_1902.xlsx")

## Remove some of the columns for the public release of the data

In [ ]:
df_impacts = pd.read_parquet(data_path / "df_impacts_per_100g_1902.parquet")

In [ ]:
columns_to_include = [ 'mean_GHG', 'sd_mean_GHG', 'mean_Land', 'sd_mean_Land', 'mean_Eut', 'sd_mean_Eut', 'mean_WaterUse', 'sd_mean_Eut',
                      'median_GHG', 'sd_median_GHG' ,'median_Land', 'sd_median_Land', 'median_Eut', 'sd_median_Eut', 'median_WaterUse', 'sd_median_WaterUse', 'median_price_sim', 'sd_price_sim']

In [ ]:
df_impacts_publication = df_impacts[columns_to_include]

In [ ]:
columns_rename = {'median_price_sim': 'price',
                  'sd_price_sim': 'sd_price'}

In [ ]:
df_impacts_publication.rename(columns=columns_rename, inplace=True)

/tmp/ipykernel_1231/4093588650.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_impacts_publication.rename(columns=columns_rename, inplace=True)


In [ ]:
ndb_data = pd.read_csv(data_path / "foodDB_data_processing/NDB_matching_data_FINAL.csv")

In [ ]:
ndb_id_mapping = ndb_data[['Local description', 'Food composition record ID']].drop_duplicates(subset=['Local description'], keep='first')
ndb_id_mapping = ndb_id_mapping.set_index('Local description')['Food composition record ID']

# Map the Food composition record ID to df_impacts_publication's index
df_impacts_publication['Food composition record ID'] = df_impacts_publication.index.map(ndb_id_mapping)

# Move the 'Food composition record ID' column to the first position as requested
food_comp_id_column = df_impacts_publication['Food composition record ID']
df_impacts_publication = df_impacts_publication.drop(columns=['Food composition record ID'])
df_impacts_publication.insert(0, 'Food composition record ID', food_comp_id_column)

/tmp/ipykernel_1231/3699687981.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_impacts_publication['Food composition record ID'] = df_impacts_publication.index.map(ndb_id_mapping)


In [ ]:
df_impacts_publication.to_excel(data_path / "NDB_foodDB_impact_data.xlsx")

In [ ]:
df_impacts_publication.to_csv(data_path / "NDB_foodDB_impact_data.csv")